# 1. NSGs, Application Security Groups, and Routing

## NSG implementation deep dive

SC-900 covered NSG basics. AZ-500 needs you to:
- Design effective rule sets
- Use Application Security Groups (ASGs) for dynamic grouping
- Implement User-Defined Routes (UDRs) for forced tunneling
- Diagnose traffic with Network Watcher

### NSG rule processing

```
Inbound traffic → Subnet NSG → NIC NSG → VM
Outbound traffic → VM → NIC NSG → Subnet NSG
```

If **both** subnet and NIC have NSGs, traffic must pass **both**. Most restrictive wins.

In [ ]:
import json

# Real-world NSG for a 3-tier application
NSG_WEB_TIER = {
    'name': 'nsg-web',
    'rules': [
        {'priority': 100, 'name': 'AllowHTTPS',         'direction': 'Inbound',  'src': 'Internet',      'dst': 'VirtualNetwork', 'port': '443',  'protocol': 'TCP', 'action': 'Allow'},
        {'priority': 110, 'name': 'AllowHTTP',          'direction': 'Inbound',  'src': 'Internet',      'dst': 'VirtualNetwork', 'port': '80',   'protocol': 'TCP', 'action': 'Allow'},
        {'priority': 120, 'name': 'AllowBastionSSH',    'direction': 'Inbound',  'src': '10.0.3.0/24',   'dst': 'VirtualNetwork', 'port': '22',   'protocol': 'TCP', 'action': 'Allow'},
        {'priority': 200, 'name': 'AllowHealthProbe',   'direction': 'Inbound',  'src': 'AzureLoadBalancer', 'dst': '*',           'port': '*',    'protocol': '*',   'action': 'Allow'},
        {'priority': 4096, 'name': 'DenyAllInbound',    'direction': 'Inbound',  'src': '*',             'dst': '*',              'port': '*',    'protocol': '*',   'action': 'Deny'},
    ],
}

NSG_APP_TIER = {
    'name': 'nsg-app',
    'rules': [
        {'priority': 100, 'name': 'AllowFromWebTier',   'direction': 'Inbound',  'src': '10.0.1.0/24',   'dst': 'VirtualNetwork', 'port': '8080', 'protocol': 'TCP', 'action': 'Allow'},
        {'priority': 120, 'name': 'AllowBastionSSH',    'direction': 'Inbound',  'src': '10.0.3.0/24',   'dst': 'VirtualNetwork', 'port': '22',   'protocol': 'TCP', 'action': 'Allow'},
        {'priority': 4096, 'name': 'DenyAllInbound',    'direction': 'Inbound',  'src': '*',             'dst': '*',              'port': '*',    'protocol': '*',   'action': 'Deny'},
    ],
}

NSG_DB_TIER = {
    'name': 'nsg-db',
    'rules': [
        {'priority': 100, 'name': 'AllowFromAppTier',   'direction': 'Inbound',  'src': '10.0.2.0/24',   'dst': 'VirtualNetwork', 'port': '5432', 'protocol': 'TCP', 'action': 'Allow'},
        {'priority': 4096, 'name': 'DenyAllInbound',    'direction': 'Inbound',  'src': '*',             'dst': '*',              'port': '*',    'protocol': '*',   'action': 'Deny'},
    ],
}

for nsg in [NSG_WEB_TIER, NSG_APP_TIER, NSG_DB_TIER]:
    print(f'=== {nsg["name"]} ===')
    for r in nsg['rules']:
        print(f'  {r["priority"]:>4} {r["action"]:<5} {r["name"]:<25} {r["src"]:<20} → port {r["port"]}')
    print()

print('Architecture: Internet → [nsg-web] Web VMs → [nsg-app] App VMs → [nsg-db] Database')
print('Each tier only accepts traffic from the tier above it.')
print('Database has NO internet access and NO SSH access (not even Bastion — only app tier).')

## Application Security Groups (ASGs)

**Problem**: when VMs scale up/down, maintaining NSG rules with IP addresses is painful.

**Solution**: ASGs let you group VMs logically, then use the group name in NSG rules instead of IP ranges.

```bash
# Create ASGs
az network asg create -g rg-prod -n asg-web-servers
az network asg create -g rg-prod -n asg-app-servers
az network asg create -g rg-prod -n asg-db-servers

# Associate a NIC with an ASG
az network nic ip-config update \
  -g rg-prod --nic-name web-vm-nic --name ipconfig1 \
  --application-security-groups asg-web-servers

# Use ASG in NSG rule (instead of IP range)
az network nsg rule create -g rg-prod --nsg-name nsg-app \
  -n AllowFromWebASG --priority 100 --direction Inbound \
  --source-asgs asg-web-servers --destination-port-ranges 8080 \
  --protocol TCP --access Allow
```

When a new web VM joins the ASG, the NSG rule automatically applies — no IP updates needed.

In [ ]:
# Simulate ASG-based NSG evaluation
ASG_MEMBERS = {
    'asg-web-servers': ['10.0.1.10', '10.0.1.11', '10.0.1.12'],
    'asg-app-servers': ['10.0.2.10', '10.0.2.11'],
    'asg-db-servers':  ['10.0.2.50'],
}

ASG_NSG_RULES = [
    {'priority': 100, 'name': 'AllowWebToApp', 'src_asg': 'asg-web-servers', 'dst_asg': 'asg-app-servers', 'port': 8080, 'action': 'Allow'},
    {'priority': 110, 'name': 'AllowAppToDB',  'src_asg': 'asg-app-servers', 'dst_asg': 'asg-db-servers',  'port': 5432, 'action': 'Allow'},
    {'priority': 4096, 'name': 'DenyAll',       'src_asg': '*',              'dst_asg': '*',               'port': '*',  'action': 'Deny'},
]

def ip_in_asg(ip, asg_name):
    if asg_name == '*':
        return True
    return ip in ASG_MEMBERS.get(asg_name, [])

def evaluate_asg_nsg(src_ip, dst_ip, port):
    for rule in sorted(ASG_NSG_RULES, key=lambda r: r['priority']):
        port_match = rule['port'] == '*' or rule['port'] == port
        src_match = ip_in_asg(src_ip, rule['src_asg'])
        dst_match = ip_in_asg(dst_ip, rule['dst_asg'])
        if port_match and src_match and dst_match:
            return f'{"✅" if rule["action"] == "Allow" else "🚫"} {rule["action"]} (rule: {rule["name"]})'
    return '🚫 Deny (implicit)'

print('=== ASG-based traffic evaluation ===\n')
tests = [
    ('Web VM → App VM:8080', '10.0.1.10', '10.0.2.10', 8080),
    ('App VM → DB:5432',     '10.0.2.10', '10.0.2.50', 5432),
    ('Web VM → DB:5432',     '10.0.1.10', '10.0.2.50', 5432),
    ('New web VM → App:8080','10.0.1.12', '10.0.2.11', 8080),
]
for desc, src, dst, port in tests:
    result = evaluate_asg_nsg(src, dst, port)
    print(f'  {desc:<30} → {result}')

print('\n💡 When 10.0.1.12 joined asg-web-servers, it automatically got access — no rule changes!')

## User-Defined Routes (UDRs)

By default, Azure routes traffic between subnets and to the internet automatically. UDRs override this.

### Common scenarios

| Scenario | UDR configuration |
|----------|-------------------|
| **Force tunnel to firewall** | 0.0.0.0/0 → Azure Firewall NVA |
| **Route to on-prem via VPN** | 10.0.0.0/8 → VPN Gateway |
| **Block internet egress** | 0.0.0.0/0 → None (drops traffic) |
| **Route between spokes via hub** | spoke2-CIDR → hub NVA |

### Forced tunneling through Azure Firewall

```bash
# Create route table
az network route-table create -g rg-prod -n rt-force-tunnel

# Add default route to Azure Firewall
az network route-table route create -g rg-prod \
  --route-table-name rt-force-tunnel -n to-firewall \
  --next-hop-type VirtualAppliance \
  --next-hop-ip-address 10.0.255.4 \
  --address-prefix 0.0.0.0/0

# Associate with subnet
az network vnet subnet update -g rg-prod --vnet-name vnet-prod \
  -n sn-app --route-table rt-force-tunnel
```

Now all traffic from `sn-app` goes through the firewall — no direct internet access.

## Network Watcher

Diagnostic tools for network issues:

| Tool | What it does |
|------|--------------|
| **IP flow verify** | Test if a packet is allowed/denied by NSGs |
| **Next hop** | Show which route a packet takes |
| **NSG flow logs** | Log all traffic through an NSG |
| **Connection troubleshoot** | End-to-end connectivity test |
| **Packet capture** | Capture packets on a VM NIC |
| **NSG diagnostics** | Show effective NSG rules |

```bash
# Test if traffic is allowed
az network watcher test-ip-flow \
  --vm my-vm -g rg-prod --direction Inbound \
  --protocol TCP --local 10.0.1.10:80 --remote 203.0.113.1:12345
```

---
## Summary

| Implementation | Key details |
|---------------|-------------|
| **NSG dual evaluation** | Subnet NSG + NIC NSG both apply; most restrictive wins |
| **ASGs** | Group VMs logically; rules auto-apply when VMs join |
| **UDRs** | Override default routing; force tunneling to firewall |
| **Network Watcher** | IP flow verify, NSG diagnostics, packet capture |

**Next**: [Notebook 2 — Private Access](02_private_access.ipynb)